# SEAS 8515 – Data Engineering for AI
## Homework 4 (Summer 2026)

## Question 1: Getting Started with Databricks
This question **must** be completed using Databricks.

Embed screenshots directly in this notebook using either Markdown or load the image and display using another Python library (such as opencv or PIL).

### Screenshot: Databricks Workspace

*(Insert screenshot of Databricks workspace here)*

### Screenshot: Uploaded learning_sparkv2.dbc Folder

*(Insert screenshot showing uploaded folder structure here)*

### Screenshot: Executed Notebook

*(Insert screenshot showing all cells executed successfully)*

## Question 2: Analyze Text Data with Spark
Use PySpark. Databricks or Google Colab is acceptable.
You must use the provided `assignment_4.txt` file.

In [ ]:
try:
    # In Databricks, spark is already available
    spark
    print("Running in Databricks")
except NameError:
    # Running in Colab or locally
    import subprocess
    subprocess.run(["pip", "install", "pyspark", "-q"], check=True)
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[*]").appName("SEAS8515_HW4").getOrCreate()
    print("SparkSession created")

spark

In [ ]:
# Load assignment_4.txt as an RDD (upload this file to your Databricks workspace or Colab)
TEXT_FILE = "assignment_4.txt"

text_rdd = spark.sparkContext.textFile(TEXT_FILE)

print("First 10 lines:")
for line in text_rdd.take(10):
    print(line)

In [ ]:
total_lines = text_rdd.count()
print(f"Total number of lines: {total_lines}")

In [ ]:
# Filter lines containing "tensor" (case-insensitive, matches tensor, Tensor, tensorflow, etc.)
tensor_lines = text_rdd.filter(lambda line: "tensor" in line.lower())

tensor_count = tensor_lines.count()
print(f"Number of lines containing 'tensor': {tensor_count}")

print("\nMatching lines:")
for line in tensor_lines.collect():
    print(line)

## Question 3: Aggregation and Filtering with Spark
Use PySpark DataFrame operations.

In [ ]:
from pyspark.sql.functions import count, desc, col

# Upload cancer_dataset_uae.csv to your workspace and update the path if needed
CANCER_FILE = "cancer_dataset_uae.csv"

df = spark.read.csv(CANCER_FILE, header=True, inferSchema=True)
print(f"Rows: {df.count()}  Columns: {len(df.columns)}")
df.printSchema()
df.show(5)

In [ ]:
# Group by Cancer Type and Outcome, count patients, sort descending
cancer_outcome = (
    df.groupBy("Cancer_Type", "Outcome")
    .agg(count("*").alias("total_patients"))
    .orderBy(desc("total_patients"))
)

print("Patient counts by Cancer Type and Outcome (descending):")
cancer_outcome.show(truncate=False)

In [ ]:
# Filter to Abu Dhabi patients, group by Treatment Type, show top 5
abu_dhabi_treatments = (
    df.filter(col("Emirate") == "Abu Dhabi")
    .groupBy("Treatment_Type")
    .agg(count("*").alias("patient_count"))
    .orderBy(desc("patient_count"))
    .limit(5)
)

print("Top 5 treatment types for Abu Dhabi patients:")
abu_dhabi_treatments.show(truncate=False)

In [ ]:
# Analyze Smoking Status vs Outcome
smoking_outcome = (
    df.groupBy("Smoking_Status", "Outcome")
    .agg(count("*").alias("patient_count"))
    .orderBy("Smoking_Status", "Outcome")
)

print("Patient counts by Smoking Status and Outcome:")
smoking_outcome.show(truncate=False)

# Calculate survival rate per smoking group for clearer comparison
from pyspark.sql.functions import round as spark_round, sum as spark_sum

total_per_group = df.groupBy("Smoking_Status").agg(count("*").alias("total"))

recovered = (
    df.filter(col("Outcome") == "Recovered")
    .groupBy("Smoking_Status")
    .agg(count("*").alias("recovered"))
)

summary = (
    total_per_group.join(recovered, "Smoking_Status", "left")
    .withColumn("recovery_rate_pct", spark_round((col("recovered") / col("total")) * 100, 1))
    .orderBy("Smoking_Status")
)

print("\nRecovery rate by Smoking Status:")
summary.show(truncate=False)